In [1]:
# Cell 1 — Mount Drive and extract DICOMs
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import tarfile

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS'
os.makedirs(DICOM_DIR, exist_ok=True)

print('Extracting DICOMs...')
with tarfile.open(f'{DATA_DIR}/fastmri_prostate_DICOMS_IDS_001_312.tar', 'r') as tar:
    tar.extractall(DICOM_DIR)

DICOM_DIR = '/content/DICOMS/DICOMS'
print(f'Patients: {len(os.listdir(DICOM_DIR))}')

Mounted at /content/drive
Extracting DICOMs...


/tmp/ipykernel_8025/3946954999.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DICOM_DIR)


Patients: 312


In [2]:
# Cell 2 — Installs
!pip install pydicom h5py -q
!pip install pydicom h5py optuna -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.3 MB/s eta 0:00:00


In [3]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import tarfile
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image
import torchvision.transforms as transforms
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Cell 4 — Load labels and create splits
DATA_DIR  = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS/DICOMS'

labels_tar = f'{DATA_DIR}/labels.tar'
with tarfile.open(labels_tar, 'r') as tar:
    f = tar.extractfile('labels/t2_slice_level_labels.csv')
    t2_labels = pd.read_csv(f)
    f = tar.extractfile('labels/volume_exam_labels.csv')
    volume_labels = pd.read_csv(f)

volume_labels['binary_label'] = (volume_labels['t2_volume_level'] >= 3).astype(int)
patient_split = t2_labels.drop_duplicates('fastmri_pt_id')[['fastmri_pt_id', 'data_split']]
df = patient_split.merge(
    volume_labels[['fastmri_pt_id', 'binary_label', 't2_volume_level']],
    on='fastmri_pt_id'
)

# Remove invalid patients with empty T2 folders
invalid_ids = [115, 258]
train_df = df[df['data_split'] == 'training'].reset_index(drop=True)
val_df   = df[df['data_split'] == 'validation'].reset_index(drop=True)
test_df  = df[df['data_split'] == 'test'].reset_index(drop=True)

train_df = train_df[~train_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
val_df   = val_df[~val_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
test_df  = test_df[~test_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(df['binary_label'].value_counts())

Train: 217, Val: 47, Test: 46
binary_label
0    178
1    134
Name: count, dtype: int64


In [5]:
# Cell 5 — MIL Dataset class (returns all 30 slices per patient)
NUM_SLICES = 30

class ProstateT2MILDataset(Dataset):
    def __init__(self, df, dicom_dir, augment=False):
        self.df = df.reset_index(drop=True)
        self.dicom_dir = dicom_dir

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

        print(f'MIL Dataset: {len(self.df)} patients')
        print(f'Labels: {self.df.binary_label.value_counts().to_dict()}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pt_id = str(row.fastmri_pt_id).zfill(3)
        t2_path = os.path.join(self.dicom_dir, pt_id, 'AX_T2')

        slices = []
        for fname in os.listdir(t2_path):
            if fname.endswith('.dcm'):
                ds = pydicom.dcmread(os.path.join(t2_path, fname))
                slices.append((int(ds.InstanceNumber), ds.pixel_array.astype(np.float32)))

        slices.sort(key=lambda x: x[0])
        volume = np.stack([s[1] for s in slices])  # (N, 320, 320)

        # Per-volume min-max normalization
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

        # Always take exactly NUM_SLICES middle slices
        n = volume.shape[0]
        mid = n // 2
        half = NUM_SLICES // 2
        start = max(0, mid - half)
        end   = min(n, start + NUM_SLICES)
        volume = volume[start:end]

        # Pad if less than NUM_SLICES
        if volume.shape[0] < NUM_SLICES:
            pad = np.zeros((NUM_SLICES - volume.shape[0], volume.shape[1], volume.shape[2]))
            volume = np.concatenate([volume, pad], axis=0)

        # Convert each slice to RGB tensor
        slice_tensors = []
        for s in volume:
            img = Image.fromarray((s * 255).astype(np.uint8))
            img = img.convert('RGB')
            img = self.transform(img)
            slice_tensors.append(img)

        volume_tensor = torch.stack(slice_tensors)  # (30, 3, 224, 224)
        label = torch.tensor(row.binary_label, dtype=torch.float32)
        return volume_tensor, label

In [6]:
# Cell 6 — MIL DataLoaders
mil_train_dataset = ProstateT2MILDataset(train_df, DICOM_DIR, augment=True)
mil_val_dataset   = ProstateT2MILDataset(val_df,   DICOM_DIR, augment=False)
mil_test_dataset  = ProstateT2MILDataset(test_df,  DICOM_DIR, augment=False)

# Small batch size — each sample is 30 images
mil_train_loader = DataLoader(mil_train_dataset, batch_size=4, shuffle=True,
                              num_workers=4, pin_memory=True, prefetch_factor=2)
mil_val_loader   = DataLoader(mil_val_dataset,   batch_size=4, shuffle=False,
                              num_workers=0)
mil_test_loader  = DataLoader(mil_test_dataset,  batch_size=4, shuffle=False,
                              num_workers=0)

# Test one sample
sample_vol, sample_label = mil_train_dataset[0]
print(f'Volume shape: {sample_vol.shape}')  # should be (30, 3, 224, 224)
print(f'Label: {sample_label}')

MIL Dataset: 217 patients
Labels: {0: 129, 1: 88}
MIL Dataset: 47 patients
Labels: {1: 26, 0: 21}
MIL Dataset: 46 patients
Labels: {0: 26, 1: 20}
Volume shape: torch.Size([30, 3, 224, 224])
Label: 0.0


In [7]:
# Cell 7 — Attention MIL Architecture
class AttentionMIL(nn.Module):
    def __init__(self, feature_dim=768):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(feature_dim, 1)
        )

    def forward(self, x):
        # x: (batch, num_slices, feature_dim)
        attn = self.attention(x)           # (batch, num_slices, 1)
        attn = torch.softmax(attn, dim=1)  # normalize across slices
        x = (attn * x).sum(dim=1)         # weighted sum -> (batch, feature_dim)
        return self.classifier(x)


class ConvNeXtMIL(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
        # Freeze first 4 stages
        for i, layer in enumerate(backbone.features):
            if i < 4:
                for param in layer.parameters():
                    param.requires_grad = False
        # Keep feature extractor only (no classifier)
        self.feature_extractor = nn.Sequential(
            backbone.features,
            backbone.avgpool
        )
        self.mil = AttentionMIL(feature_dim=768)

    def forward(self, x):
        # x: (batch, num_slices, 3, 224, 224)
        batch, num_slices, C, H, W = x.shape
        x = x.view(batch * num_slices, C, H, W)
        features = self.feature_extractor(x)          # (batch*slices, 768, 1, 1)
        features = features.view(batch, num_slices, -1)  # (batch, slices, 768)
        return self.mil(features)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Test forward pass
test_model = ConvNeXtMIL().to(device)
test_input = torch.randn(2, 30, 3, 224, 224).to(device)
test_output = test_model(test_input)
print(f'Output shape: {test_output.shape}')  # should be (2, 1)
del test_model, test_input, test_output

Using device: cuda
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:01<00:00, 172MB/s]


Output shape: torch.Size([2, 1])


In [8]:
# Cell 8 — Train/Val functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc

In [9]:
import optuna

n_neg = (train_df['binary_label'] == 0).sum()
n_pos = (train_df['binary_label'] == 1).sum()

def mil_objective(trial):
    lr_backbone  = trial.suggest_float('lr_backbone',  1e-7, 1e-4, log=True)
    lr_head      = trial.suggest_float('lr_head',      1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1,  log=True)

    trial_model = ConvNeXtMIL().to(device)

    trial_backbone_params = [p for p in trial_model.parameters()
                             if p.requires_grad and id(p) not in
                             set(id(p) for p in trial_model.mil.parameters())]
    trial_head_params = list(trial_model.mil.parameters())

    optimizer = torch.optim.AdamW([
        {'params': trial_backbone_params, 'lr': lr_backbone},
        {'params': trial_head_params,     'lr': lr_head}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    pw        = torch.tensor([n_neg / n_pos]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_val_auc     = 0
    patience         = 3
    patience_counter = 0

    for epoch in range(10):
        train_epoch(trial_model, mil_train_loader, optimizer, criterion, device)
        scheduler.step()
        _, val_auc = val_epoch(trial_model, mil_val_loader, criterion, device)

        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_auc

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
)
study.optimize(mil_objective, n_trials=10, timeout=7200)

print('\nBest MIL trial:')
print(f'  Val AUC: {study.best_value:.4f}')
print(f'  Params: {study.best_params}')

[I 2026-05-18 00:12:00,832] A new study created in memory with name: no-name-3139132c-c475-4460-adb4-48e71b56c34c
[I 2026-05-18 00:16:11,789] Trial 0 finished with value: 0.5567765567765568 and parameters: {'lr_backbone': 9.848612589279854e-06, 'lr_head': 4.7212319181351964e-05, 'weight_decay': 0.021391610145663376}. Best is trial 0 with value: 0.5567765567765568.
[I 2026-05-18 00:22:18,115] Trial 1 finished with value: 0.4249084249084249 and parameters: {'lr_backbone': 5.688800232858828e-07, 'lr_head': 2.1738790254247444e-05, 'weight_decay': 0.0012538761657953739}. Best is trial 0 with value: 0.5567765567765568.
[I 2026-05-18 00:27:17,273] Trial 2 finished with value: 0.4908424908424908 and parameters: {'lr_backbone': 5.7773263131276944e-05, 'lr_head': 0.00037746916064720036, 'weight_decay': 0.0023698457234369594}. Best is trial 0 with value: 0.5567765567765568.
[I 2026-05-18 00:31:20,772] Trial 3 finished with value: 0.47985347985347987 and parameters: {'lr_backbone': 4.6967226038174


Best MIL trial:
  Val AUC: 0.5568
  Params: {'lr_backbone': 9.848612589279854e-06, 'lr_head': 4.7212319181351964e-05, 'weight_decay': 0.021391610145663376}


In [10]:
best = study.best_params
print(f'Training final MIL model with: {best}')

mil_model = ConvNeXtMIL().to(device)

mil_backbone_params = [p for p in mil_model.parameters()
                       if p.requires_grad and id(p) not in
                       set(id(p) for p in mil_model.mil.parameters())]
mil_head_params = list(mil_model.mil.parameters())

mil_optimizer = torch.optim.AdamW([
    {'params': mil_backbone_params, 'lr': best['lr_backbone']},
    {'params': mil_head_params,     'lr': best['lr_head']}
], weight_decay=best['weight_decay'])

mil_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(mil_optimizer, T_max=50)
mil_pos_weight = torch.tensor([n_neg / n_pos]).to(device)
mil_criterion  = nn.BCEWithLogitsLoss(pos_weight=mil_pos_weight)
print('Ready')

Training final MIL model with: {'lr_backbone': 9.848612589279854e-06, 'lr_head': 4.7212319181351964e-05, 'weight_decay': 0.021391610145663376}
Ready


In [11]:
# Cell 10 — Training loop
EPOCHS   = 50
PATIENCE = 10
best_mil_val_auc    = 0
patience_counter    = 0
best_mil_model_path = f'{DATA_DIR}/best_mil_model.pth'

for epoch in range(EPOCHS):
    train_loss, train_auc = train_epoch(mil_model, mil_train_loader,
                                        mil_optimizer, mil_criterion, device)
    mil_scheduler.step()
    val_loss, val_auc = val_epoch(mil_model, mil_val_loader,
                                  mil_criterion, device)

    if val_auc > best_mil_val_auc:
        best_mil_val_auc = val_auc
        patience_counter = 0
        torch.save(mil_model.state_dict(), best_mil_model_path)
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} *** BEST ***')
    else:
        patience_counter += 1
        print(f'Epoch {epoch+1:02d}/{EPOCHS} — Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}.')
            break

print(f'\nBest MIL Val AUC: {best_mil_val_auc:.4f}')

Epoch 01/50 — Train Loss: 0.8217 AUC: 0.5257 | Val Loss: 0.8639 AUC: 0.6154 *** BEST ***
Epoch 02/50 — Train Loss: 0.8234 AUC: 0.5231 | Val Loss: 0.8848 AUC: 0.5751 (patience 1/10)
Epoch 03/50 — Train Loss: 0.8210 AUC: 0.5612 | Val Loss: 0.8839 AUC: 0.5385 (patience 2/10)
Epoch 04/50 — Train Loss: 0.8239 AUC: 0.5566 | Val Loss: 0.8800 AUC: 0.5330 (patience 3/10)
Epoch 05/50 — Train Loss: 0.8148 AUC: 0.5868 | Val Loss: 0.8633 AUC: 0.5385 (patience 4/10)
Epoch 06/50 — Train Loss: 0.8168 AUC: 0.6008 | Val Loss: 0.8727 AUC: 0.5055 (patience 5/10)
Epoch 07/50 — Train Loss: 0.8116 AUC: 0.6135 | Val Loss: 0.9191 AUC: 0.4982 (patience 6/10)
Epoch 08/50 — Train Loss: 0.8063 AUC: 0.6357 | Val Loss: 0.8854 AUC: 0.4982 (patience 7/10)
Epoch 09/50 — Train Loss: 0.8038 AUC: 0.6090 | Val Loss: 0.8322 AUC: 0.4927 (patience 8/10)
Epoch 10/50 — Train Loss: 0.7827 AUC: 0.6583 | Val Loss: 0.9142 AUC: 0.4835 (patience 9/10)
Epoch 11/50 — Train Loss: 0.7645 AUC: 0.6978 | Val Loss: 0.9524 AUC: 0.4817 (patien

In [12]:
# Cell 11 — Test evaluation
mil_model.load_state_dict(torch.load(best_mil_model_path))
_, mil_test_auc = val_epoch(mil_model, mil_test_loader, mil_criterion, device)
print(f'MIL Test AUC: {mil_test_auc:.4f}')

MIL Test AUC: 0.4115


In [13]:
# Save MIL results
import json

mil_results = {
    'model': 'ConvNeXt-Small + Attention MIL',
    'approach': 'All 30 slices with attention pooling',
    'best_val_auc': float(best_mil_val_auc),
    'test_auc': float(mil_test_auc),
    'best_params': study.best_params,
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'epochs_trained': epoch + 1,
    'num_slices': NUM_SLICES,
}

results_path = f'{DATA_DIR}/mil_results.json'
with open(results_path, 'w') as f:
    json.dump(mil_results, f, indent=2)

print('MIL results saved:')
print(json.dumps(mil_results, indent=2))

MIL results saved:
{
  "model": "ConvNeXt-Small + Attention MIL",
  "approach": "All 30 slices with attention pooling",
  "best_val_auc": 0.6153846153846154,
  "test_auc": 0.4115384615384615,
  "best_params": {
    "lr_backbone": 9.848612589279854e-06,
    "lr_head": 4.7212319181351964e-05,
    "weight_decay": 0.021391610145663376
  },
  "n_train": 217,
  "n_val": 47,
  "n_test": 46,
  "epochs_trained": 11,
  "num_slices": 30
}
